# Import Data from CSV to PostgreSQL DB

## Prerequisites

Before running this notebook, ensure:

1. **PostgreSQL is installed and running** on your system
2. **Database exists**: A database named `edward_local` should exist  
   - Create it using: `CREATE DATABASE edward_local;` in PostgreSQL
3. **Python packages installed**: 
   - `pandas`
   - `psycopg2-binary`
   
   Install with: `pip install pandas psycopg2-binary`

4. **CSV files**: The `olist_data/` folder should contain all 9 CSV files

---

## What This Notebook Does

This notebook provides a function to:
- Create a PostgreSQL schema named **`olist_db`**
- Create all necessary tables in the schema with proper data types
- Import data from CSV files in the `olist_data/` folder
- Set up foreign key relationships between tables
- Create indexes for optimized query performance
- Provide a summary of imported data

All tables will be created in the `olist_db` schema for better organization.

---

In [1]:
import pandas as pd
import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
import os
from pathlib import Path
from typing import Dict, Optional


def import_csv_to_postgresql(
    db_name: str = "edward_local",
    host: str = "localhost",
    port: int = 5432,
    user: str = "edward_user",
    password: str = "0421051853",
    data_dir: Optional[str] = None,
    schema_name: str = "olist_db"
):
    """
    Import CSV data from olist_data directory to PostgreSQL database.
    
    Args:
        db_name: Name of the PostgreSQL database (default: edward_local)
        host: PostgreSQL server host (default: localhost)
        port: PostgreSQL server port (default: 5432)
        user: PostgreSQL username (default: postgres)
        password: PostgreSQL password
        data_dir: Path to CSV data directory (default: ./olist_data)
        schema_name: PostgreSQL schema name (default: olist_db)
    
    Returns:
        bool: True if import successful, False otherwise
    """
    
    # Set data directory
    if data_dir is None:
        data_dir = Path(__file__).parent / "olist_data" if '__file__' in globals() else Path("olist_data")
    else:
        data_dir = Path(data_dir)
    
    # Mapping of table names to CSV files
    datasets = {
        'customers': 'olist_customers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'order_payments': 'olist_order_payments_dataset.csv',
        'order_reviews': 'olist_order_reviews_dataset.csv',
        'orders': 'olist_orders_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'product_category_translation': 'product_category_name_translation.csv'
    }
    
    # Define table schemas with proper PostgreSQL data types
    table_schemas = {
        'customers': """
            CREATE TABLE IF NOT EXISTS {schema}.customers (
                customer_id VARCHAR(50) PRIMARY KEY,
                customer_unique_id VARCHAR(50),
                customer_zip_code_prefix VARCHAR(10),
                customer_city VARCHAR(100),
                customer_state VARCHAR(2)
            )
        """,
        'geolocation': """
            CREATE TABLE IF NOT EXISTS {schema}.geolocation (
                id SERIAL PRIMARY KEY,
                geolocation_zip_code_prefix VARCHAR(10),
                geolocation_lat NUMERIC(10, 7),
                geolocation_lng NUMERIC(10, 7),
                geolocation_city VARCHAR(100),
                geolocation_state VARCHAR(2)
            )
        """,
        'orders': """
            CREATE TABLE IF NOT EXISTS {schema}.orders (
                order_id VARCHAR(50) PRIMARY KEY,
                customer_id VARCHAR(50),
                order_status VARCHAR(20),
                order_purchase_timestamp TIMESTAMP,
                order_approved_at TIMESTAMP,
                order_delivered_carrier_date TIMESTAMP,
                order_delivered_customer_date TIMESTAMP,
                order_estimated_delivery_date TIMESTAMP,
                FOREIGN KEY (customer_id) REFERENCES {schema}.customers(customer_id)
            )
        """,
        'products': """
            CREATE TABLE IF NOT EXISTS {schema}.products (
                product_id VARCHAR(50) PRIMARY KEY,
                product_category_name VARCHAR(100),
                product_name_lenght NUMERIC(10, 2),
                product_description_lenght NUMERIC(10, 2),
                product_photos_qty NUMERIC(10, 2),
                product_weight_g NUMERIC(10, 2),
                product_length_cm NUMERIC(10, 2),
                product_height_cm NUMERIC(10, 2),
                product_width_cm NUMERIC(10, 2)
            )
        """,
        'sellers': """
            CREATE TABLE IF NOT EXISTS {schema}.sellers (
                seller_id VARCHAR(50) PRIMARY KEY,
                seller_zip_code_prefix VARCHAR(10),
                seller_city VARCHAR(100),
                seller_state VARCHAR(2)
            )
        """,
        'order_items': """
            CREATE TABLE IF NOT EXISTS {schema}.order_items (
                id SERIAL PRIMARY KEY,
                order_id VARCHAR(50),
                order_item_id VARCHAR(20),
                product_id VARCHAR(50),
                seller_id VARCHAR(50),
                shipping_limit_date TIMESTAMP,
                price NUMERIC(10, 2),
                freight_value NUMERIC(10, 2),
                FOREIGN KEY (order_id) REFERENCES {schema}.orders(order_id),
                FOREIGN KEY (product_id) REFERENCES {schema}.products(product_id),
                FOREIGN KEY (seller_id) REFERENCES {schema}.sellers(seller_id)
            )
        """,
        'order_payments': """
            CREATE TABLE IF NOT EXISTS {schema}.order_payments (
                id SERIAL PRIMARY KEY,
                order_id VARCHAR(50),
                payment_sequential NUMERIC(10, 1),
                payment_type VARCHAR(20),
                payment_installments NUMERIC(10, 1),
                payment_value NUMERIC(10, 2),
                FOREIGN KEY (order_id) REFERENCES {schema}.orders(order_id)
            )
        """,
        'order_reviews': """
            CREATE TABLE IF NOT EXISTS {schema}.order_reviews (
                review_id VARCHAR(50) PRIMARY KEY,
                order_id VARCHAR(50),
                review_score NUMERIC(10, 2),
                review_comment_title TEXT,
                review_comment_message TEXT,
                review_creation_date TIMESTAMP,
                review_answer_timestamp TIMESTAMP,
                FOREIGN KEY (order_id) REFERENCES {schema}.orders(order_id)
            )
        """,
        'product_category_translation': """
            CREATE TABLE IF NOT EXISTS {schema}.product_category_translation (
                product_category_name VARCHAR(100) PRIMARY KEY,
                product_category_name_english VARCHAR(100)
            )
        """
    }
    
    # Order of table creation (respecting foreign key dependencies)
    table_order = [
        'customers',
        'orders',
        'products',
        'sellers',
        'geolocation',
        'product_category_translation',
        'order_items',
        'order_payments',
        'order_reviews'
    ]
    
    # Tables with natural primary keys (used for ON CONFLICT handling)
    pk_columns = {
        'customers': 'customer_id',
        'orders': 'order_id',
        'products': 'product_id',
        'sellers': 'seller_id',
        'order_reviews': 'review_id',
        'product_category_translation': 'product_category_name'
    }
    
    conn = None
    cursor = None
    
    try:
        # Connect to PostgreSQL
        print(f"Connecting to PostgreSQL database '{db_name}' at {host}:{port}...")
        conn = psycopg2.connect(
            dbname=db_name,
            user=user,
            password=password,
            host=host,
            port=port
        )
        cursor = conn.cursor()
        print("✓ Connected successfully!\n")
        
        # Create schema if it doesn't exist
        print(f"Creating schema '{schema_name}' if it doesn't exist...")
        cursor.execute(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(sql.Identifier(schema_name)))
        conn.commit()
        print(f"✓ Schema '{schema_name}' ready\n")
        
        # Set search path to use the schema
        cursor.execute(sql.SQL("SET search_path TO {}, public").format(sql.Identifier(schema_name)))
        
        # Create tables in order
        print("Creating tables...")
        for table_name in table_order:
            if table_name in table_schemas:
                # Replace {schema} placeholder with actual schema name
                create_sql = table_schemas[table_name].replace('{schema}', schema_name)
                cursor.execute(create_sql)
                print(f"  ✓ Created/verified table '{schema_name}.{table_name}'")
        conn.commit()
        print()
        
        # Clear existing data in schema using TRUNCATE in reverse dependency order
        print(f"Clearing existing data in schema '{schema_name}' (reverse dependency order)...")
        for tbl in reversed(table_order):
            cursor.execute(sql.SQL("TRUNCATE TABLE {}.{} RESTART IDENTITY CASCADE").format(
                sql.Identifier(schema_name), sql.Identifier(tbl)
            ))
        conn.commit()
        print("✓ Cleared existing data\n")
        
        # Import data from CSVs
        print("Importing data from CSV files...")
        print("=" * 60)
        
        for table_name in table_order:
            if table_name not in datasets:
                continue
                
            csv_file = datasets[table_name]
            csv_path = data_dir / csv_file
            
            if not csv_path.exists():
                print(f"⚠ Warning: {csv_file} not found, skipping...")
                continue
            
            print(f"\nProcessing {csv_file}...")
            
            # Read CSV file
            df = pd.read_csv(csv_path)
            print(f"  - Rows: {len(df):,}")
            print(f"  - Columns: {list(df.columns)}")
            
            # Prepare column names
            columns = list(df.columns)
            
            # Build insert query with optional ON CONFLICT for PK tables
            if table_name in pk_columns and pk_columns[table_name] in columns:
                pk_col = pk_columns[table_name]
                insert_query = sql.SQL("INSERT INTO {}.{} ({}) VALUES ({}) ON CONFLICT ({}) DO NOTHING").format(
                    sql.Identifier(schema_name),
                    sql.Identifier(table_name),
                    sql.SQL(', ').join(map(sql.Identifier, columns)),
                    sql.SQL(', ').join(sql.Placeholder() * len(columns)),
                    sql.Identifier(pk_col)
                )
            else:
                insert_query = sql.SQL("INSERT INTO {}.{} ({}) VALUES ({})").format(
                    sql.Identifier(schema_name),
                    sql.Identifier(table_name),
                    sql.SQL(', ').join(map(sql.Identifier, columns)),
                    sql.SQL(', ').join(sql.Placeholder() * len(columns))
                )
            
            # Convert DataFrame to list of tuples
            # Replace NaN values with None for proper NULL handling
            records = df.where(pd.notnull(df), None).values.tolist()
            
            # Batch insert
            batch_size = 1000
            total_inserted = 0
            
            for i in range(0, len(records), batch_size):
                batch = records[i:i + batch_size]
                cursor.executemany(insert_query, batch)
                total_inserted += len(batch)
                if i % 10000 == 0 and i > 0:
                    print(f"  - Inserted {total_inserted:,} rows so far...")
            
            conn.commit()
            print(f"  ✓ Imported {total_inserted:,} rows to '{schema_name}.{table_name}'")
        
        # Create indexes for better query performance
        print("\n" + "=" * 60)
        print("Creating indexes...")
        
        indexes = [
            f"CREATE INDEX IF NOT EXISTS idx_customers_unique_id ON {schema_name}.customers(customer_unique_id)",
            f"CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON {schema_name}.orders(customer_id)",
            f"CREATE INDEX IF NOT EXISTS idx_orders_status ON {schema_name}.orders(order_status)",
            f"CREATE INDEX IF NOT EXISTS idx_order_items_order_id ON {schema_name}.order_items(order_id)",
            f"CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON {schema_name}.order_items(product_id)",
            f"CREATE INDEX IF NOT EXISTS idx_order_items_seller_id ON {schema_name}.order_items(seller_id)",
            f"CREATE INDEX IF NOT EXISTS idx_order_payments_order_id ON {schema_name}.order_payments(order_id)",
            f"CREATE INDEX IF NOT EXISTS idx_order_reviews_order_id ON {schema_name}.order_reviews(order_id)",
            f"CREATE INDEX IF NOT EXISTS idx_geolocation_zip ON {schema_name}.geolocation(geolocation_zip_code_prefix)"
        ]
        
        for index_sql in indexes:
            cursor.execute(index_sql)
            index_name = index_sql.split('idx_')[1].split(' ON')[0]
            print(f"  ✓ {index_name}")
        
        conn.commit()
        
        # Show summary
        print("\n" + "=" * 60)
        print("DATABASE IMPORT SUMMARY")
        print("=" * 60)
        
        for table_name in table_order:
            cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                sql.Identifier(schema_name),
                sql.Identifier(table_name)
            ))
            count = cursor.fetchone()[0]
            print(f"  {schema_name}.{table_name:30s}: {count:,} rows")
        
        print(f"\n✓ Data import completed successfully into schema '{schema_name}'!")
        return True
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        if conn:
            conn.rollback()
        return False
        
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()
            print("\n✓ Database connection closed.")


# Example usage:
# import_csv_to_postgresql(
#     db_name="edward_local",
#     user="postgres",
#     password="your_password",
#     host="localhost",
#     schema_name="olist_db"
# )

In [2]:
# Run the import function
# Data will be imported into the 'olist_db' schema

success = import_csv_to_postgresql(
    db_name="edward_local",
    user="edward_user",  # Update with your PostgreSQL username if different
    password="0421051853",      # Add your PostgreSQL password here
    host="localhost",
    port=5432,
    data_dir="../olist_data",  # Path relative to notebook location
    schema_name="olist_db"  # Schema where tables will be created
)

if success:
    print("\nAll data has been successfully imported to the database!")
else:
    print("\nImport failed. Please check the error messages above.")

Connecting to PostgreSQL database 'edward_local' at localhost:5432...
✓ Connected successfully!

Creating schema 'olist_db' if it doesn't exist...
✓ Schema 'olist_db' ready

Creating tables...
  ✓ Created/verified table 'olist_db.customers'
  ✓ Created/verified table 'olist_db.orders'
  ✓ Created/verified table 'olist_db.products'
  ✓ Created/verified table 'olist_db.sellers'
  ✓ Created/verified table 'olist_db.geolocation'
  ✓ Created/verified table 'olist_db.product_category_translation'
  ✓ Created/verified table 'olist_db.order_items'
  ✓ Created/verified table 'olist_db.order_payments'
  ✓ Created/verified table 'olist_db.order_reviews'

Clearing existing data in schema 'olist_db' (reverse dependency order)...
✓ Cleared existing data

Importing data from CSV files...

Processing olist_customers_dataset.csv...
  - Rows: 99,441
  - Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
  - Inserted 11,000 rows so far...
  - Ins

## Verify Import (Optional)

After importing, you can verify the data was imported correctly by connecting to the database and running a few queries:

In [3]:
# Verify data import with sample queries
import psycopg2
import pandas as pd

# Update these connection parameters to match your setup
conn = psycopg2.connect(
    dbname="edward_local",
    user="edward_user",
    password="0421051853",
    host="localhost",
    port=5432
)

# Set search path to use the olist_db
cursor = conn.cursor()
cursor.execute("SET search_path TO olist_db, public")
cursor.close()

# Test query 1: Count records in each table
print("Record counts in olist_db:")
print("=" * 40)
tables = ['customers', 'orders', 'products', 'sellers', 'geolocation', 
          'order_items', 'order_payments', 'order_reviews', 'product_category_translation']

for table in tables:
    df = pd.read_sql(f"SELECT COUNT(*) as count FROM olist_db.{table}", conn)
    print(f"{table:30s}: {df['count'][0]:,}")

print("\n" + "=" * 40)

# Test query 2: Sample data from customers table
print("\nSample customers data:")
df_customers = pd.read_sql("SELECT * FROM olist_db.customers LIMIT 5", conn)
display(df_customers)

# Test query 3: Sample data from orders table
print("\nSample orders data:")
df_orders = pd.read_sql("SELECT * FROM olist_db.orders LIMIT 5", conn)
display(df_orders)

conn.close()
print("\n✓ Verification complete!")

Record counts in olist_db:
customers                     : 99,441
orders                        : 99,441
products                      : 32,951
sellers                       : 3,095
geolocation                   : 1,000,163
order_items                   : 112,650
order_payments                : 103,886
order_reviews                 : 98,410
product_category_translation  : 71


Sample customers data:


/tmp/ipykernel_74044/3099107087.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT COUNT(*) as count FROM olist_db.{table}", conn)
/tmp/ipykernel_74044/3099107087.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customers = pd.read_sql("SELECT * FROM olist_db.customers LIMIT 5", conn)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Sample orders data:


/tmp/ipykernel_74044/3099107087.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_orders = pd.read_sql("SELECT * FROM olist_db.orders LIMIT 5", conn)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26



✓ Verification complete!
